In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv('data/GSE115469_Data.csv', index_col=0)

In [3]:
data.shape

(20007, 8444)

In [4]:
data.isna().values.any()

np.False_

# PRETPROCESIRANJE - Opšti koraci

In [5]:
gene_frequency = (data > 0).mean(axis=1)
data = data.loc[gene_frequency >= 0.05]

In [6]:
data.shape

(6717, 8444)

In [7]:
gene_var = data.var(axis=1)
gene_var_sorted = gene_var.sort_values(ascending=False)
gene_var_keep = gene_var_sorted[:2000]

print(gene_var_keep)

APOC3     12.340822
APOA2     11.126836
ORM1       9.849425
HP         9.393121
ALB        9.389534
            ...    
TACC1      0.375015
PDXDC1     0.374714
FUNDC2     0.374669
RNF167     0.374545
PSAT1      0.374325
Length: 2000, dtype: float64


In [8]:
data = data.loc[gene_var_keep.index]
data.shape

(2000, 8444)

In [9]:
data = data.clip(upper=data.quantile(0.999).max())

In [10]:
data = data.T
data.shape

(8444, 2000)

# PRETPROCESIRANJE - Pravila pridruživanja

In [11]:
data.head(10)

,APOC3,APOA2,ORM1,HP,ALB,APOC1,TTR,SAA1,APOA1,FGA,...,HMGCR,RNF149,GIMAP1,APH1A,MAF1,TACC1,PDXDC1,FUNDC2,RNF167,PSAT1
P1TLH_AAACCTGAGCAGCCTC_1,0.836165,0.836165,0.000000,1.362102,0.000000,0.836165,0.000000,1.362102,0.836165,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.836165,0.0,0.000000,0.0,0.0
P1TLH_AAACCTGTCCTCATTA_1,1.436498,1.436498,3.458904,1.149924,4.797971,2.059859,0.572995,2.493720,1.300315,1.675472,...,0.0,0.000000,0.000000,0.314760,0.0,0.982011,0.0,0.000000,0.0,0.0
P1TLH_AAACCTGTCTAAGCCA_1,1.820614,0.000000,0.000000,1.180248,0.000000,0.000000,1.180248,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,1.180248,0.0,1.180248,0.0,0.0
P1TLH_AAACGGGAGTAGGCCA_1,0.000000,0.644905,0.000000,0.000000,0.000000,0.000000,0.000000,1.428094,0.000000,0.000000,...,0.0,1.428094,0.644905,0.644905,0.0,0.000000,0.0,0.000000,0.0,0.0
P1TLH_AAACGGGGTTCGGGCT_1,1.284221,1.284221,0.780522,1.656843,1.656843,0.780522,0.000000,1.656843,0.780522,0.000000,...,0.0,0.000000,1.284221,0.780522,0.0,0.000000,0.0,0.000000,0.0,0.0
P1TLH_AAAGCAACAGTAAGAT_1,0.000000,2.216831,0.000000,2.216831,2.216831,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,2.216831,0.0,0.0
P1TLH_AAAGCAAGTCGCGTGT_1,3.072206,1.512300,1.512300,2.234284,1.512300,0.000000,2.234284,2.234284,1.512300,0.000000,...,0.0,1.512300,2.234284,0.000000,0.0,0.000000,0.0,1.512300,0.0,0.0
P1TLH_AAAGCAAGTGTTTGTG_1,1.172884,0.876891,1.172884,1.172884,1.628089,0.504068,0.000000,2.252131,0.876891,0.504068,...,0.0,0.876891,0.504068,0.504068,0.0,0.504068,0.0,0.000000,0.0,0.0
P1TLH_AAAGCAAGTTGATTCG_1,2.126138,0.000000,2.456096,2.456096,1.085305,1.697618,0.000000,3.610965,2.126138,1.697618,...,0.0,0.000000,1.085305,0.000000,0.0,1.085305,0.0,0.000000,0.0,0.0
P1TLH_AAAGTAGCAGACGTAG_1,0.775766,1.649062,1.649062,0.775766,1.277507,1.277507,0.000000,2.889257,1.277507,0.775766,...,0.0,0.000000,0.775766,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0


## Korak 1: Binarizacija
Za istraživanje pravila pridruživanja potreban nam je skup podataka sa vrednostima 1 i 0, odnosno True i False. Ovde ćemo dodeljivati vrednosti True (gen je prisutan u ćeliji) i False (gen nije prisutan) u odnosu na granicu specifičnu za svaku ćeliju. Biramo ovaj pristup da bismo smanjili uticaj razlike u rasponu količine ekspresije među različitim genima u različitim ćelijama.

In [12]:
gene_thresholds = data.quantile(0.75, axis=1)
binary_AR_data = data.gt(gene_thresholds, axis=0).astype(bool)

In [13]:
binary_AR_data.head(10)

,APOC3,APOA2,ORM1,HP,ALB,APOC1,TTR,SAA1,APOA1,FGA,...,HMGCR,RNF149,GIMAP1,APH1A,MAF1,TACC1,PDXDC1,FUNDC2,RNF167,PSAT1
P1TLH_AAACCTGAGCAGCCTC_1,False,False,False,True,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
P1TLH_AAACCTGTCCTCATTA_1,True,True,True,True,True,True,False,True,True,True,...,False,False,False,False,False,True,False,False,False,False
P1TLH_AAACCTGTCTAAGCCA_1,True,False,False,True,False,False,True,False,False,False,...,False,False,False,False,False,True,False,True,False,False
P1TLH_AAACGGGAGTAGGCCA_1,False,False,False,False,False,False,False,True,False,False,...,False,True,False,False,False,False,False,False,False,False
P1TLH_AAACGGGGTTCGGGCT_1,True,True,False,True,True,False,False,True,False,False,...,False,False,True,False,False,False,False,False,False,False
P1TLH_AAAGCAACAGTAAGAT_1,False,True,False,True,True,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
P1TLH_AAAGCAAGTCGCGTGT_1,True,False,False,True,False,False,True,True,False,False,...,False,False,True,False,False,False,False,False,False,False
P1TLH_AAAGCAAGTGTTTGTG_1,True,False,True,True,True,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
P1TLH_AAAGCAAGTTGATTCG_1,True,False,True,True,False,True,False,True,True,True,...,False,False,False,False,False,False,False,False,False,False
P1TLH_AAAGTAGCAGACGTAG_1,False,True,True,False,True,True,False,True,True,False,...,False,False,False,False,False,False,False,False,False,False


## Korak 2: Uklanjanje previše čestih/retkih gena
Geni koji se eksprimuju u skoro svim ćelijama proizvode trivijalna pravila, dok oni koji se ne eksprimuju skoro ni u jednoj ćeliji mogu proizvesti neuverljiva pravila.

Granice za odsecanje gena se mogu podešavati u odnosu na potrebnu količinu odsecanja.

In [14]:
gene_frequency = binary_AR_data.mean(axis=0)

print((gene_frequency > 0.90).sum())
print((gene_frequency < 0.06).sum())

64
552


In [15]:
binary_AR_data.shape

(8444, 2000)

In [16]:
binary_AR_data = binary_AR_data.loc[:, (gene_frequency >= 0.06) & (gene_frequency <= 0.90)]
binary_AR_data.shape

(8444, 1384)

# Apriori + Association_rules
### Generisanje čestih skupova stavki i izdvajanje pravila pridruživanja (za sve ćelije)
Ovu metodu koristimo za pronalaženje interesantnih pravila **globalne koekspresije gena** u našem skupu podataka. Apriori prvo izdvaja grupe gena koji se često eksprimuju zajedno (česti skupovi stavki -- gena). Nakon toga generator pravila pridruživanja izvodi pravila oblika *GeneA → GeneB* ili *GeneA, GeneB → GeneC*. Ta pravila nam govore sledeće: kada se eksprimuju geni iz glave pravila (leva strana pravila -- anticedent), često se takođe eksprimuju geni iz tela pravila (desna strana -- consequent).

## Dodatno pretprocesiranje: Odabir atributa (Feature Selection)
Trenutni broj gena (stavki) u našem skupu je 1384, što je veoma veliki broj za analizu čestih skupova stavki jer lako dolazi do kombinatorne eksplozije. Zbog toga koristimo već pomenutu metodu zadržavanja *n* najvarijabilnijih gena. Ovde uzimamo vrednost *n*=500.

In [17]:
gene_var = binary_AR_data.var(axis=0)
gene_var_sorted = gene_var.sort_values(ascending=False)
gene_var_keep = gene_var_sorted[:500]

print(gene_var_keep)

ITM2B     0.250030
DDT       0.250029
C3        0.250026
DBI       0.250022
COX5B     0.250019
            ...   
EIF3L     0.141314
COTL1     0.141235
SMIM14    0.140923
EIF4B     0.140923
RPL22     0.140923
Length: 500, dtype: float64


In [18]:
apriori_data = binary_AR_data.loc[:, gene_var_keep.index]
apriori_data.shape

(8444, 500)

## Primena alata
Kod poziva alata apriori podešavamo naredne parametre:
- **min_support=0.5**: Min_support je prag podrške skupa stavki - skup stavki je čest ako ima podršku veću od praga. Ova vrednost znači da skup gena mora biti prisutan u barem 50% ćelija da bi se smatrao čestim. Ovako visok prag značajno umanjuje broj skupova kandidata, čineći apriori izvršivim nad velikim brojem transakcija (8444), dok u fokusu ostaju najrobustniji obrasci koekspresije.
- **max_len=3**: Max_len je gornje ograničenje broja elemenata skupova stavki. Postavljen je na 3 zato što je to zlatna sredina za biološku interpretaciju kasnije izvedenih pravila pridruživanja (koekspresija 3 gena), a smanjuje kompleksnost izračunavanja, čime se ubrzava izvršavanje. 

In [19]:
from mlxtend.frequent_patterns import apriori, association_rules

In [20]:
freq_itemsets = apriori(
     apriori_data,
     min_support=0.5,
     use_colnames=True,
     max_len=3,
     verbose=1,
     low_memory=True
)

Processing 6308 combinations | Sampling itemset size 3


In [21]:
freq_itemsets.shape

(2369, 2)

Možemo da izdvojimo skupove sa određenim brojem stavki, npr. 3.

In [22]:
multi_itemsets = freq_itemsets[freq_itemsets['itemsets'].apply(lambda x: len(x)) == 3]
multi_itemsets.shape

(1577, 2)

In [23]:
multi_itemsets.head(10)

,support,itemsets
792,0.503079,"(HLA-A, TMSB10, TMSB4X)"
793,0.511843,"(HLA-A, HLA-B, TMSB4X)"
794,0.502369,"(APOE, APOC3, ORM1)"
795,0.505329,"(APOE, MT2A, ORM1)"
796,0.502369,"(MT1X, APOC3, ORM1)"
797,0.511606,"(MT1X, MT2A, APOC3)"
798,0.510066,"(MT1X, MT2A, ORM1)"
799,0.504737,"(RPL10A, PFDN5, RPL14)"
800,0.512435,"(TMSB4X, PFDN5, RPL14)"
801,0.506040,"(RPL14, RPL7A, PFDN5)"


In [24]:
def print_rules(rules):
    for index, row in rules.iterrows():
        antecedent = ', '.join(list(row['antecedents']))
        consequent = ', '.join(list(row['consequents']))
        print(f"{{{antecedent}}} --> {{{consequent}}}")


Pravila pridruživanja mogu se generisati za različite metrike. Ovde pokrećemo association_rules za metriku poverenja (confidence) i lift mere.

**Confidence**: Pravila su generisana i filtrirana na osnovu poverenja, koje za transakcije koje sadrže glavu pravila meri udeo onih koja sadrže i telo pravila. Pravila sa visokim poverenjem su pokazatelj jakih prediktivnih veza. Poverenje stavlja akcenat na pouzdanost pravila.

**Lift**: Pravila su generisana i filtrirana na osnovu lift mere, koja meri koliko se češće glava i telo pravila pojavljuju zajedno nego što bismo očekivali da su statistički nezavisni. Lift mera veća od 1 kazuje nam da među genima postoji pozitivna zavisnost. Lift mera stavlja akcenat na interesantnost pravila.

In [25]:
rules_conf = association_rules(
    freq_itemsets,
    metric="confidence",
    min_threshold=0.9
)

In [26]:
print(len(rules_conf))
print_rules(rules_conf)

1703
{FGL1} --> {ORM1}
{AMBP} --> {APOC3}
{AMBP} --> {ORM1}
{AMBP} --> {MT2A}
{EEF1B2} --> {RPS5}
{EEF1B2} --> {RPL14}
{EEF1B2} --> {RPL10A}
{EEF1B2} --> {TMSB4X}
{EEF1B2} --> {RPL7A}
{EEF1B2} --> {RPL24}
{EEF1B2} --> {RPL22}
{FGA} --> {APOC3}
{FGA} --> {ORM1}
{FGA} --> {MT2A}
{HLA-A} --> {TMSB10}
{HLA-A} --> {HLA-B}
{HLA-A} --> {TMSB4X}
{HLA-A} --> {RPL7A}
{FGG} --> {APOC3}
{FGG} --> {ORM1}
{FGG} --> {MT2A}
{USMG5} --> {COX7C}
{H3F3B} --> {RPL14}
{H3F3B} --> {RPL10A}
{H3F3B} --> {TMSB4X}
{H3F3B} --> {RPL7A}
{SEPP1} --> {MT2A}
{APOE} --> {APOC3}
{APOE} --> {ORM1}
{APOE} --> {MT2A}
{MT1X} --> {APOC3}
{MT1X} --> {ORM1}
{MT1X} --> {MT2A}
{PFDN5} --> {TMSB4X}
{PFDN5} --> {RPL22}
{FGB} --> {ORM1}
{HLA-C} --> {HLA-B}
{HLA-C} --> {TMSB4X}
{MT1G} --> {APOC3}
{MT1G} --> {ORM1}
{MT1G} --> {MT2A}
{RPL4} --> {RPL14}
{RPL4} --> {RPL7A}
{RPL4} --> {RPL24}
{RPL4} --> {RPL22}
{TMSB10} --> {HLA-B}
{TMSB10} --> {TMSB4X}
{RPSA} --> {RPL14}
{RPSA} --> {RPL10A}
{RPSA} --> {TMSB4X}
{RPSA} --> {RPL7A}
{APOC1

In [27]:
rules_lift = association_rules(
    freq_itemsets,
    metric="lift",
    min_threshold=1.2
)

In [28]:
print(len(rules_lift))
print_rules(rules_lift)

696
{FGL1} --> {ORM1}
{ORM1} --> {FGL1}
{AMBP} --> {APOC3}
{APOC3} --> {AMBP}
{AMBP} --> {ORM1}
{ORM1} --> {AMBP}
{FGA} --> {APOC3}
{APOC3} --> {FGA}
{HLA-A} --> {TMSB10}
{TMSB10} --> {HLA-A}
{HLA-A} --> {HLA-B}
{HLA-B} --> {HLA-A}
{APOC3} --> {FGG}
{FGG} --> {APOC3}
{APOE} --> {APOC1}
{APOC1} --> {APOE}
{APOE} --> {APOA1}
{APOA1} --> {APOE}
{APOE} --> {APOC3}
{APOC3} --> {APOE}
{MT1X} --> {MT1G}
{MT1G} --> {MT1X}
{MT1X} --> {APOC1}
{APOC1} --> {MT1X}
{MT1X} --> {APOA1}
{APOA1} --> {MT1X}
{MT1X} --> {APOC3}
{APOC3} --> {MT1X}
{APOA2} --> {FGB}
{FGB} --> {APOA2}
{FGB} --> {APOC1}
{APOC1} --> {FGB}
{FGB} --> {APOA1}
{APOA1} --> {FGB}
{FGB} --> {APOC3}
{APOC3} --> {FGB}
{TMSB10} --> {HLA-C}
{HLA-C} --> {TMSB10}
{HLA-B} --> {HLA-C}
{HLA-C} --> {HLA-B}
{MT1G} --> {APOA2}
{APOA2} --> {MT1G}
{MT1G} --> {APOC1}
{APOC1} --> {MT1G}
{MT1G} --> {APOA1}
{APOA1} --> {MT1G}
{MT1G} --> {APOC3}
{APOC3} --> {MT1G}
{TMSB10} --> {RPL36A}
{RPL36A} --> {TMSB10}
{TMSB10} --> {HLA-B}
{HLA-B} --> {TMSB10}
{APO

Poverenje često favorizuje pravila koja sadrže česte gene, dok lift mera otkriva specifičnije i potencijalno biološki informativnije veze između gena. Poređenje ova dva skupa pravila pomaže nam da razdvojimo opšte pouzdane obrasce koekspresije od neobično jakih povezanosti.

# FP-Growth + Association_rules
### Generisanje čestih skupova stavki i izdvajanje pravila pridruživanja (za svaki ćelijski tip posebno)

Ovu metodu koristimo za pronalaženje interesantnih pravila **"lokalne" koekspresije gena**, na nivou jednog ćelijskog tipa. U te svrhe uvodimo drugi skup podataka, pomoću kog povezujemo ćelije sa njihovim tipovima. Algoritam FP-Growth primenjen je nad transakcijama (ćelijama) zadatog tipa *cell_type*. Nakon toga generator pravila pridruživanja izvodi pravila koja opisuju veze između prisutnih gena za konkretan tip ćelije. Ovaj pristup otkriva nam obrasce specifične za jedan ćelijski tip, iz kojih možemo da izvučemo važne zaključke o ustaljenim modulima genske ekspresije.

## Dodatno pretprocesiranje: Feature Selection, spajanje tabela

Ponovo vršimo smanjenje dimenzionalnosti skupa radi omogućavanja izvršivosti analize. Ovaj korak pretprocesiranja primenjujemo odvojeno za različite alate jer nemaju svi oni isti "prag tolerancije" - neki mogu podržati više jedinstvenih stavki od drugih.

In [29]:
gene_var = binary_AR_data.var(axis=0)
gene_var_sorted = gene_var.sort_values(ascending=False)
gene_var_keep = gene_var_sorted[:800]

print(gene_var_keep)

ITM2B    0.250030
DDT      0.250029
C3       0.250026
DBI      0.250022
COX5B    0.250019
           ...   
HSPD1    0.100410
ADH1C    0.100410
PSMA2    0.100410
C1QB     0.100044
GNAI2    0.099952
Length: 800, dtype: float64


In [30]:
fpgrowth_data = binary_AR_data.loc[:, gene_var_keep.index]
fpgrowth_data.shape

(8444, 800)

In [31]:
cell_types = pd.read_csv('data/GSE115469_CellClusterType.txt', sep='\t')

In [32]:
cell_types

,CellName,Sample,Cell#,Cluster#,CellType
0,P1TLH_AAACCTGAGCAGCCTC_1,P1TLH,AAACCTGAGCAGCCTC,12,Central_venous_LSECs
1,P1TLH_AAACCTGTCCTCATTA_1,P1TLH,AAACCTGTCCTCATTA,17,Cholangiocytes
2,P1TLH_AAACCTGTCTAAGCCA_1,P1TLH,AAACCTGTCTAAGCCA,12,Central_venous_LSECs
3,P1TLH_AAACGGGAGTAGGCCA_1,P1TLH,AAACGGGAGTAGGCCA,10,Non-inflammatory_Macrophage
4,P1TLH_AAACGGGGTTCGGGCT_1,P1TLH,AAACGGGGTTCGGGCT,2,alpha-beta_T_Cells
...,...,...,...,...,...
8439,P5TLH_TTTGTCAGTGTTCTTT_1,P5TLH,TTTGTCAGTGTTCTTT,17,Cholangiocytes
8440,P5TLH_TTTGTCAGTTTAGGAA_1,P5TLH,TTTGTCAGTTTAGGAA,11,Periportal_LSECs
8441,P5TLH_TTTGTCATCAGCTTAG_1,P5TLH,TTTGTCATCAGCTTAG,17,Cholangiocytes
8442,P5TLH_TTTGTCATCCACGCAG_1,P5TLH,TTTGTCATCCACGCAG,4,Inflammatory_Macrophage


Uklanjamo redundantne kolone iz tabele.

In [33]:
cell_types = cell_types.drop(columns=['Sample', 'Cell#', 'Cluster#'])

In [34]:
fpgrowth_data.index.name = 'CellName'
fpgrowth_data.reset_index(inplace=True)

fpgrowth_data

,CellName,ITM2B,DDT,C3,DBI,COX5B,HPX,RBP4,GPX1,PEBP1,...,KRT10,PRRC2C,GTF3A,LSM5,TKT,HSPD1,ADH1C,PSMA2,C1QB,GNAI2
0,P1TLH_AAACCTGAGCAGCCTC_1,True,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
1,P1TLH_AAACCTGTCCTCATTA_1,True,False,True,True,True,False,True,False,True,...,False,False,False,False,False,False,False,False,False,False
2,P1TLH_AAACCTGTCTAAGCCA_1,True,True,False,True,True,False,False,True,False,...,True,False,False,False,False,False,False,False,False,False
3,P1TLH_AAACGGGAGTAGGCCA_1,True,False,False,False,False,False,False,True,False,...,False,False,False,True,True,False,False,False,True,True
4,P1TLH_AAACGGGGTTCGGGCT_1,True,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8439,P5TLH_TTTGTCAGTGTTCTTT_1,True,True,True,True,True,False,True,False,True,...,True,True,False,False,False,False,False,False,False,False
8440,P5TLH_TTTGTCAGTTTAGGAA_1,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
8441,P5TLH_TTTGTCATCAGCTTAG_1,True,True,False,True,False,False,True,False,True,...,False,False,False,False,False,False,False,False,False,False
8442,P5TLH_TTTGTCATCCACGCAG_1,False,False,False,True,True,False,False,True,False,...,True,False,False,False,True,False,False,True,False,True


Spajamo dve tabele podataka po koloni CellName.

In [35]:
merged_data = pd.merge(
    fpgrowth_data,
    cell_types,
    on='CellName'
)

In [36]:
merged_data.set_index('CellName', inplace=True)
merged_data.index.name = ''

merged_data

,ITM2B,DDT,C3,DBI,COX5B,HPX,RBP4,GPX1,PEBP1,BTF3,...,PRRC2C,GTF3A,LSM5,TKT,HSPD1,ADH1C,PSMA2,C1QB,GNAI2,CellType
,,,,,,,,,,,,,,,,,,,,,
P1TLH_AAACCTGAGCAGCCTC_1,True,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,Central_venous_LSECs
P1TLH_AAACCTGTCCTCATTA_1,True,False,True,True,True,False,True,False,True,False,...,False,False,False,False,False,False,False,False,False,Cholangiocytes
P1TLH_AAACCTGTCTAAGCCA_1,True,True,False,True,True,False,False,True,False,True,...,False,False,False,False,False,False,False,False,False,Central_venous_LSECs
P1TLH_AAACGGGAGTAGGCCA_1,True,False,False,False,False,False,False,True,False,True,...,False,False,True,True,False,False,False,True,True,Non-inflammatory_Macrophage
P1TLH_AAACGGGGTTCGGGCT_1,True,False,False,False,False,False,False,False,False,True,...,True,False,False,False,False,False,False,False,True,alpha-beta_T_Cells
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
P5TLH_TTTGTCAGTGTTCTTT_1,True,True,True,True,True,False,True,False,True,True,...,True,False,False,False,False,False,False,False,False,Cholangiocytes
P5TLH_TTTGTCAGTTTAGGAA_1,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,Periportal_LSECs
P5TLH_TTTGTCATCAGCTTAG_1,True,True,False,True,False,False,True,False,True,True,...,False,False,False,False,False,False,False,False,False,Cholangiocytes


In [37]:
merged_data.shape

(8444, 801)

## Primena algoritama

Inicijalno je pokušana primena metode za svaki od 20 ćelijskih tipova kroz for-each petlju. Međutim, to nije bilo izvodljivo zbog progresivnog porasta kompleksnosti izračunavanja. Za neke ćelijske tipove odabrani prag podrške proizveo bi ogroman broj čestih skupova, što vodi do veoma dugog izvršavanja, koje ne može biti gotovo u razumnom vremenu. Zato je implementirana funkcija za pokretanje analize nad zadatim tipom ćelije, koja kao argumente prihvata i pragove podrške i poverenja kako bi analiza bila prilagodljiva različitim tipovima ćelija. Pre poziva samog alata fpgrowth izvršeno je još jedno filtriranje gena radi dodatnog smanjenja prostora pretrage.

Parametri alata fpgrowth slični su onima koje smo videli kod apriori.

In [38]:
from mlxtend.frequent_patterns import fpgrowth

In [39]:
results = {}

In [40]:
def rules_for_cell_type(cell_type, data, results, min_sup, min_conf):
    subset = data[data['CellType'] == cell_type]
    transactions = subset.drop(columns='CellType')
    n_cells = len(transactions)

    min_support = max(20 / n_cells, min_sup)

    gene_supports = transactions.mean(axis=0)

    transactions = transactions.loc[
        :,
        (gene_supports >= 0.3) &
        (gene_supports <= 0.8)
    ]

    top_genes = transactions.var().nlargest(100).index
    transactions = transactions[top_genes]

    freq_itemsets = fpgrowth(
        transactions,
        min_support=min_support,
        use_colnames=True,
        max_len=3
    )

    if freq_itemsets.empty:
        print(f"No frequent itemsets for cell type {cell_type}.")
    else:
        print(f"Finished frequent itemset generation for cell type {cell_type}.")

    rules = association_rules(
        freq_itemsets,
        metric='confidence',
        min_threshold=min_conf
    )

    print(f"Finished rules generation for cell type {cell_type}.")

    results[cell_type] = {
        'itemsets': freq_itemsets,
        'rules': rules
    }

In [41]:
celltype = 'Inflammatory_Macrophage'
rules_for_cell_type(celltype, merged_data, results, min_sup=0.2, min_conf=0.7)

Finished frequent itemset generation for cell type Inflammatory_Macrophage.
Finished rules generation for cell type Inflammatory_Macrophage.


In [42]:
print(len(results[celltype]['itemsets']))
print(len(results[celltype]['rules']))

10151
4462


In [43]:
results[celltype]['rules'].sort_values("confidence", ascending=False).head(20)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1858,"(C1QB, NBEAL1)",(C1QA),0.233702,0.559656,0.232472,0.994737,1.777409,1.0,0.101680,83.665437,0.570775,0.414474,0.988048,0.705061
3588,"(C1QB, HLA-DRB1)",(C1QA),0.403444,0.559656,0.399754,0.990854,1.770470,1.0,0.173964,48.144321,0.729485,0.709607,0.979229,0.852570
3686,"(C1QB, ALDOA)",(C1QA),0.264453,0.559656,0.261993,0.990698,1.770192,1.0,0.113990,47.337023,0.591518,0.466083,0.978875,0.729415
3003,"(C1QB, EIF3F)",(C1QA),0.227552,0.559656,0.225092,0.989189,1.767496,1.0,0.097741,40.731857,0.562145,0.400438,0.975449,0.695693
1880,"(C1QB, FXYD5)",(C1QA),0.202952,0.559656,0.200492,0.987879,1.765155,1.0,0.086909,36.328413,0.543854,0.356674,0.972473,0.673060
2903,"(C1QB, NDUFA4)",(C1QA),0.257073,0.559656,0.253383,0.985646,1.761165,1.0,0.109510,30.677327,0.581745,0.449782,0.967403,0.719197
3753,"(HLA-DPA1, C1QB)",(C1QA),0.415744,0.559656,0.409594,0.985207,1.760381,1.0,0.176921,29.767282,0.739301,0.723913,0.966406,0.858538
1736,"(ATP5D, C1QB)",(C1QA),0.217712,0.559656,0.214022,0.983051,1.756528,1.0,0.092178,25.980320,0.550558,0.379913,0.961509,0.682734
3582,"(C1QB, HLA-DPB1)",(C1QA),0.430504,0.559656,0.423124,0.982857,1.756182,1.0,0.182190,25.686757,0.756078,0.746204,0.961069,0.869451
3674,"(C1QB, BTG1)",(C1QA),0.285363,0.559656,0.280443,0.982759,1.756006,1.0,0.120738,25.539975,0.602440,0.496732,0.960846,0.741929


Dakle, za određeni tip ćelije, pravilo *GeneA → GeneB* govori nam da se *GeneA* i *GeneB* često zajedno eksprimuju.

# FP-Growth + Association_rules
### Generisanje čestih skupova stavki i izdvajanje pravila pridruživanja, posmatrajući CellType kao jednu od stavki

Ovaj pristup uvodi malo drugačiji pogled na pravila pridruživanja - uvođenjem tipa ćelije kao dodatne stavke (zapravo, 20 njih) efektivno možemo pronaći **pravila za klasifikaciju** ćelija u tipove. Dovoljno je samo transformisati skup tako da ćelijski tipovi postanu binarne kolone, izvesti pravila pridruživanja, a zatim tragati za pravilima koja kao telo imaju neki ćelijski tip (pravila oblika *GeneA, GeneB → CellTypeX*). Takva pravila tumačimo na sledeći način: Za pravilo *GeneA, GeneB → CellTypeX*, kombinacija eksprimovanih gena *GeneA* i *GeneB* karakteriše tip ćelije *CellTypeX*.

## Dodatno pretprocesiranje: spajanje tabela, one-hot encoding za CellType

Zbog filtriranja gena u odnosu na granicu specifičnu za svaki gen sa početka ove sveske, čak i veoma informativni geni često završe sa podrškom od samo nekoliko procenata. Zato ćemo pokušati sa drugim načinom filtriranja, ne bismo li sprečili tu pojavu.

Računaćemo da je gen eksprimovan ako je količina njegove ekspresije različita od 0.

In [44]:
fpgrowth_data = (data > 0).astype(bool)
fpgrowth_data.shape

(8444, 2000)

Nakon toga ćemo zadržati samo one gene sa "normalnom" podrškom (ovo nisu egzaktne granice, već eksperimentalne).

In [45]:
gene_supports = fpgrowth_data.mean(axis=0)

fpgrowth_data = fpgrowth_data.loc[
    :,
    (gene_supports >= 0.2) &
    (gene_supports <= 0.5)
]

fpgrowth_data.shape

(8444, 1293)

In [46]:
fpgrowth_data.index.name = 'CellName'
fpgrowth_data.reset_index(inplace=True)

fpgrowth_data

,CellName,IGKC,CRP,ANG,TAT,APCS,MT1H,HBB,SPINK1,ALDOB,...,SPPL2A,RAB4A,UBE2I,RAD21,CNDP2,PABPN1,PTRHD1,SEPT2,APH1A,RNF167
0,P1TLH_AAACCTGAGCAGCCTC_1,False,True,False,False,False,False,False,False,False,...,False,False,True,False,False,True,False,True,False,False
1,P1TLH_AAACCTGTCCTCATTA_1,True,True,True,False,True,False,False,False,False,...,True,False,True,False,True,False,True,False,True,False
2,P1TLH_AAACCTGTCTAAGCCA_1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,P1TLH_AAACGGGAGTAGGCCA_1,True,False,False,False,False,False,False,False,True,...,True,False,False,False,False,False,False,True,True,False
4,P1TLH_AAACGGGGTTCGGGCT_1,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8439,P5TLH_TTTGTCAGTGTTCTTT_1,True,True,True,False,True,False,False,False,True,...,False,False,True,False,False,True,True,False,False,True
8440,P5TLH_TTTGTCAGTTTAGGAA_1,True,False,True,False,True,False,True,False,False,...,False,False,True,False,False,True,False,False,False,False
8441,P5TLH_TTTGTCATCAGCTTAG_1,True,False,False,True,True,False,True,False,False,...,True,True,False,False,False,False,False,True,True,False
8442,P5TLH_TTTGTCATCCACGCAG_1,True,True,False,False,False,True,True,False,False,...,False,False,False,False,False,False,False,False,False,False


In [47]:
merged_data = pd.merge(
    fpgrowth_data,
    cell_types,
    on='CellName'
)

In [48]:
merged_data.set_index('CellName', inplace=True)
merged_data.index.name = ''

merged_data

,IGKC,CRP,ANG,TAT,APCS,MT1H,HBB,SPINK1,ALDOB,APOB,...,RAB4A,UBE2I,RAD21,CNDP2,PABPN1,PTRHD1,SEPT2,APH1A,RNF167,CellType
,,,,,,,,,,,,,,,,,,,,,
P1TLH_AAACCTGAGCAGCCTC_1,False,True,False,False,False,False,False,False,False,False,...,False,True,False,False,True,False,True,False,False,Central_venous_LSECs
P1TLH_AAACCTGTCCTCATTA_1,True,True,True,False,True,False,False,False,False,True,...,False,True,False,True,False,True,False,True,False,Cholangiocytes
P1TLH_AAACCTGTCTAAGCCA_1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,Central_venous_LSECs
P1TLH_AAACGGGAGTAGGCCA_1,True,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,True,True,False,Non-inflammatory_Macrophage
P1TLH_AAACGGGGTTCGGGCT_1,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,True,True,False,alpha-beta_T_Cells
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
P5TLH_TTTGTCAGTGTTCTTT_1,True,True,True,False,True,False,False,False,True,True,...,False,True,False,False,True,True,False,False,True,Cholangiocytes
P5TLH_TTTGTCAGTTTAGGAA_1,True,False,True,False,True,False,True,False,False,False,...,False,True,False,False,True,False,False,False,False,Periportal_LSECs
P5TLH_TTTGTCATCAGCTTAG_1,True,False,False,True,True,False,True,False,False,False,...,True,False,False,False,False,False,True,True,False,Cholangiocytes


In [49]:
merged_data.shape

(8444, 1294)

Ovo je način da izvršimo one-hot encoding kolone CellType.

In [50]:
celltype_dummies = pd.get_dummies(
    merged_data['CellType'],
    prefix='CellType'
).astype(bool)

In [51]:
merged_data = merged_data.drop(columns='CellType')

In [52]:
merged_with_cell_types = pd.concat(
    [merged_data, celltype_dummies],
    axis=1
)

merged_with_cell_types.shape

(8444, 1313)

In [53]:
merged_with_cell_types.head(10)

,IGKC,CRP,ANG,TAT,APCS,MT1H,HBB,SPINK1,ALDOB,APOB,...,CellType_Inflammatory_Macrophage,CellType_Mature_B_Cells,CellType_NK-like_Cells,CellType_Non-inflammatory_Macrophage,CellType_Periportal_LSECs,CellType_Plasma_Cells,CellType_Portal_endothelial_Cells,CellType_alpha-beta_T_Cells,CellType_gamma-delta_T_Cells_1,CellType_gamma-delta_T_Cells_2
,,,,,,,,,,,,,,,,,,,,,
P1TLH_AAACCTGAGCAGCCTC_1,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
P1TLH_AAACCTGTCCTCATTA_1,True,True,True,False,True,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
P1TLH_AAACCTGTCTAAGCCA_1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
P1TLH_AAACGGGAGTAGGCCA_1,True,False,False,False,False,False,False,False,True,False,...,False,False,False,True,False,False,False,False,False,False
P1TLH_AAACGGGGTTCGGGCT_1,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
P1TLH_AAAGCAACAGTAAGAT_1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
P1TLH_AAAGCAAGTCGCGTGT_1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
P1TLH_AAAGCAAGTGTTTGTG_1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
P1TLH_AAAGCAAGTTGATTCG_1,False,False,False,False,False,False,False,False,False,True,...,True,False,False,False,False,False,False,False,False,False


## Primena algoritama

In [54]:
celltype_cols = [
    c for c in merged_with_cell_types.columns
    if c.startswith("CellType_")
]

merged_with_cell_types[celltype_cols].mean().sort_values(ascending=False)

CellType_Hepatocyte_1                   0.119138
CellType_alpha-beta_T_Cells             0.113809
CellType_Hepatocyte_2                   0.107650
CellType_Inflammatory_Macrophage        0.096281
CellType_Hepatocyte_3                   0.074491
CellType_Hepatocyte_4                   0.071412
CellType_Plasma_Cells                   0.060516
CellType_NK-like_Cells                  0.057793
CellType_gamma-delta_T_Cells_1          0.054950
CellType_Non-inflammatory_Macrophage    0.044884
CellType_Periportal_LSECs               0.038726
CellType_Central_venous_LSECs           0.036239
CellType_Portal_endothelial_Cells       0.024988
CellType_Hepatocyte_5                   0.023922
CellType_Hepatocyte_6                   0.018001
CellType_Mature_B_Cells                 0.015277
CellType_Cholangiocytes                 0.014093
CellType_gamma-delta_T_Cells_2          0.012435
CellType_Erythroid_Cells                0.011014
CellType_Hepatic_Stellate_Cells         0.004382
dtype: float64

Odavde zaključujemo da podrška skupova koji sadrže CellType stavke ne može biti veća od 0.12.

Eksperimentalno je zaključeno da je ~1300 gena prevelik broj i da se mora drastično smanjiti da bi izvršavanje u razumnom vremenu bilo moguće. Stoga ćemo izdvojiti 100 najvarijabilnijih gena.

In [55]:
gene_cols = [col for col in merged_with_cell_types.columns
             if col not in celltype_cols]

gene_variances = merged_with_cell_types[gene_cols].var()
top_100_genes = gene_variances.nlargest(100).index

merged_reduced = merged_with_cell_types[
    list(top_100_genes) + celltype_cols
]

print(merged_reduced.shape)

(8444, 120)


In [56]:
freq_itemsets = fpgrowth(
    merged_reduced,
    min_support=0.1,
    use_colnames=True,
    max_len=2
)

print(f"Found {len(freq_itemsets)} frequent itemsets.")

Found 5060 frequent itemsets.


In [57]:
rules = association_rules(
    freq_itemsets,
    metric="confidence",
    min_threshold=0.7
)

print(f"Found {len(rules)} rules.")

Found 496 rules.


Pokušajmo da izdvojimo samo pravila koja nas interesuju, a to su ona koja kao telo imaju neki ćelijski tip.

In [58]:
rules_celltype = rules[rules['consequents'].apply(
                                    lambda x: any(item.startswith("CellType_")
                                                  for item in x)
                                                )
                      ]

In [59]:
len(rules_celltype)

0

U našem konkretnom primeru ne dobijamo tražene rezultate jer su CellType stavke previše retke u odnosu na izabranu vrednost minimalne podrške (0.1). S druge strane, smanjivanjem vrednosti minimalne podrške vreme izvršavanja se znatno povećava, a to u našem slučaju nije prihvatljivo.

# Charm
### Generisanje zatvorenih čestih skupova stavki za zadati ćelijski tip

**Zatvoren čest skup stavki** je čest skup stavki koji je zatvoren i ima podršku veću ili jednaku minimalnoj podršci *min_sup*. Skup stavki je **zatvoren** u skupu podataka ako ne postoji njegov nadskup koji ima istu podršku kao on.

Zatvoreni česti skupovi stavki generisani su korišćenjem alata CHARM nad listom transakcija da bi se identifikovale grupe gena koji se često zajedno javljaju u ćelijama. Cilj ove analize bio je otkrivanje neredundantnih modula koekspresije gena uklanjanjem (suvišnih) skupova stavki, koji imaju podršku jednaku većim nadskupovima. Skupovi su generisani za konkretan ćelijski tip radi stvaranja jasnije slike o pravilnostima unutar jednog tipa.

## Dodatno pretprocesiranje: pretvaranje Data Frame u listu transakcija

In [60]:
binary_AR_data.shape

(8444, 1384)

In [61]:
charm_data = binary_AR_data

In [62]:
charm_data.index.name = 'CellName'
charm_data.reset_index(inplace=True)

charm_data

,CellName,APOC3,APOA2,ORM1,HP,ALB,APOC1,TTR,APOA1,FGA,...,ANXA11,IL2RG,METTL9,SLA,LYAR,C10orf54,HLA-DMA,C7,GIMAP1,TACC1
0,P1TLH_AAACCTGAGCAGCCTC_1,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,P1TLH_AAACCTGTCCTCATTA_1,True,True,True,True,True,True,False,True,True,...,False,False,False,False,False,False,False,False,False,True
2,P1TLH_AAACCTGTCTAAGCCA_1,True,False,False,True,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,True
3,P1TLH_AAACGGGAGTAGGCCA_1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,P1TLH_AAACGGGGTTCGGGCT_1,True,True,False,True,True,False,False,False,False,...,False,True,False,True,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8439,P5TLH_TTTGTCAGTGTTCTTT_1,True,True,True,True,True,True,True,False,True,...,False,False,False,False,False,False,False,False,False,False
8440,P5TLH_TTTGTCAGTTTAGGAA_1,True,False,False,False,False,False,False,True,False,...,True,False,False,False,False,False,False,True,False,False
8441,P5TLH_TTTGTCATCAGCTTAG_1,True,False,True,True,True,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
8442,P5TLH_TTTGTCATCCACGCAG_1,True,True,False,True,True,True,False,False,False,...,True,False,False,False,False,False,False,False,False,False


In [63]:
merged_data = pd.merge(
    charm_data,
    cell_types,
    on='CellName'
)

In [64]:
merged_data.set_index('CellName', inplace=True)
merged_data.index.name = ''

merged_data

,APOC3,APOA2,ORM1,HP,ALB,APOC1,TTR,APOA1,FGA,SAA2,...,IL2RG,METTL9,SLA,LYAR,C10orf54,HLA-DMA,C7,GIMAP1,TACC1,CellType
,,,,,,,,,,,,,,,,,,,,,
P1TLH_AAACCTGAGCAGCCTC_1,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,Central_venous_LSECs
P1TLH_AAACCTGTCCTCATTA_1,True,True,True,True,True,True,False,True,True,True,...,False,False,False,False,False,False,False,False,True,Cholangiocytes
P1TLH_AAACCTGTCTAAGCCA_1,True,False,False,True,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,True,Central_venous_LSECs
P1TLH_AAACGGGAGTAGGCCA_1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,Non-inflammatory_Macrophage
P1TLH_AAACGGGGTTCGGGCT_1,True,True,False,True,True,False,False,False,False,False,...,True,False,True,False,False,False,False,True,False,alpha-beta_T_Cells
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
P5TLH_TTTGTCAGTGTTCTTT_1,True,True,True,True,True,True,True,False,True,True,...,False,False,False,False,False,False,False,False,False,Cholangiocytes
P5TLH_TTTGTCAGTTTAGGAA_1,True,False,False,False,False,False,False,True,False,True,...,False,False,False,False,False,False,True,False,False,Periportal_LSECs
P5TLH_TTTGTCATCAGCTTAG_1,True,False,True,True,True,False,False,True,False,True,...,False,False,False,False,False,False,False,False,False,Cholangiocytes


In [65]:
merged_data.shape

(8444, 1385)

Tip *Inflammatory_Macrophage* dat je kao primer, ali umesto njega može stajati bilo koji od ostalih 19 tipova.

In [66]:
cell_type = "Inflammatory_Macrophage"

subset = merged_data[merged_data['CellType'] == cell_type]
subset = subset.drop(columns='CellType')

subset.head(10)

,APOC3,APOA2,ORM1,HP,ALB,APOC1,TTR,APOA1,FGA,SAA2,...,ANXA11,IL2RG,METTL9,SLA,LYAR,C10orf54,HLA-DMA,C7,GIMAP1,TACC1
,,,,,,,,,,,,,,,,,,,,,
P1TLH_AAAGCAAGTTGATTCG_1,True,False,True,True,False,True,False,True,True,True,...,False,False,True,False,False,False,False,False,False,False
P1TLH_AAAGTAGTCTGCGGCA_1,False,True,True,True,True,False,False,False,False,True,...,False,False,False,False,False,True,False,False,False,False
P1TLH_AACCATGCAGCGAACA_1,True,True,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
P1TLH_AACTCCCAGATCCCGC_1,False,False,True,False,False,False,False,False,False,True,...,False,False,False,False,False,True,True,False,False,False
P1TLH_AACTCTTTCAACGAAA_1,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,True,False,False,True,False
P1TLH_AAGCCGCAGCTAGTGG_1,False,True,True,True,False,False,False,False,False,False,...,False,False,False,True,False,True,False,False,False,False
P1TLH_AAGGAGCGTCTCGTTC_1,True,True,False,False,True,False,False,False,False,True,...,False,False,False,False,False,True,True,False,False,False
P1TLH_AAGGTTCAGCGAAGGG_1,True,False,False,True,True,False,False,False,False,True,...,False,False,False,False,True,True,False,False,False,False
P1TLH_AAGGTTCAGTGGACGT_1,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [67]:
gene_supports = subset.mean(axis=0)

keep = (
    (gene_supports >= 0.10) &
    (gene_supports <= 0.50)
)

subset_keep = subset.loc[:, keep]

print(subset_keep.shape)

(813, 592)


In [68]:
gene_var = subset_keep.var(axis=0)

top_genes = gene_var.nlargest(100).index

subset_keep = subset_keep[top_genes]

print(subset_keep.shape)

(813, 100)


CHARM ne radi sa Data Frames, već traži listu transakcija.

In [69]:
transactions = [
    list(subset_keep.columns[row == 1])
    for _, row in subset_keep.iterrows()
]

In [70]:
pami_df = pd.DataFrame({
    "Transactions": [
        "\t".join(t)
        for t in transactions
    ]
})

In [71]:
pami_df.head(10)

,Transactions
0,NPM1\tAPOC1\tGMFG\tAPOA1\tCOX8A\tTIMP1\tANXA1\...
1,PYCARD\tITGB2\tANXA1\tHSP90AA1\tCOX5B\tHMGB1\t...
2,GMFG\tBLVRB\tCOX8A\tTIMP1\tFCGRT\tCOX5B\tHMGB1...
3,NPM1\tPYCARD\tGMFG\tBLVRB\tTIMP1\tFCGRT\tITGB2...
4,PYCARD\tGMFG\tBLVRB\tCOX8A\tTIMP1\tFCGRT\tITGB...
5,NPM1\tGMFG\tBLVRB\tTIMP1\tFCGRT\tCOX5B\tHMGB1\...
6,NPM1\tPYCARD\tBLVRB\tCOX8A\tTIMP1\tITGB2\tHSP9...
7,PYCARD\tANXA1\tHSP90AA1\tCOX5B\tMAFB\tSDCBP\tL...
8,PYCARD\tGMFG\tCOX5B\tLAMTOR4\tSLC25A5\tPCBP1\t...
9,NPM1\tAPOC1\tGMFG\tBLVRB\tCOX8A\tTIMP1\tANXA1\...


## Primena algoritma

In [72]:
from PAMI.frequentPattern.closed import CHARM as charm

In [73]:
obj = charm.CHARM(
    pami_df,
    minSup=0.1,
    sep="\t"
)

obj.mine()

Closed Frequent patterns were generated successfully using CHARM algorithm


In [74]:
patterns = obj.getPatterns()
len(patterns)

7911

Za vrednost *minSup=0.1* dobijamo ~7900 zatvorenih čestih skupova, što je previše za interpretaciju. Pokušaćemo da povećamo *minSup* i nađemo "soft spot" za dalje istraživanje.

In [75]:
for s in [0.45, 0.40, 0.35]:
    obj = charm.CHARM(pami_df, minSup=s, sep="\t")
    obj.mine()
    patterns = obj.getPatterns()
    print(f"{s}: {len(patterns)}")

Closed Frequent patterns were generated successfully using CHARM algorithm
0.45: 25
Closed Frequent patterns were generated successfully using CHARM algorithm
0.4: 61
Closed Frequent patterns were generated successfully using CHARM algorithm
0.35: 99


Prag od 0.45 daje nam premalo skupova, ali pragovi od 0.4 i 0.35 pružaju sasvim dovoljno da možemo da donesemo neke zaključke.

In [76]:
obj = charm.CHARM(
    pami_df,
    minSup=0.4,
    sep="\t"
)

obj.mine()
patterns_04 = obj.getPatterns()

Closed Frequent patterns were generated successfully using CHARM algorithm


In [77]:
print(patterns_04)

{'NOP10\t': 339, 'NBEAL1\t': 349, 'GLUL\t': 333, 'PSMA7\t': 328, 'NDUFA4\t': 344, 'UQCRQ\t': 333, 'SLC25A3\t': 340, 'RBM3\t': 344, 'HCLS1\t': 332, 'POLR2L\t': 330, 'COX7B\t': 335, 'IFITM3\t': 347, 'NAP1L1\t': 346, 'USMG5\t': 352, 'ASAH1\t': 335, 'MT-ND6\t': 354, 'CSTB\t': 327, 'ARPC5\t': 362, 'RNASE6\t': 367, 'EDF1\t': 342, 'HIGD2A\t': 326, 'COX7A2\t': 350, 'UBL5\t': 344, 'AP2S1\t': 342, 'FXYD5\t': 347, 'JUN\t': 334, 'GLTSCR2\t': 345, 'ATP5D\t': 358, 'ANXA2\t': 345, 'CD68\t': 337, 'ZFP36L1\t': 337, 'CORO1A\t': 356, 'TIMP1\t': 388, 'EMP3\t': 372, 'CHCHD2\t': 355, 'EVI2B\t': 355, 'LAMTOR4\t': 378, 'PCBP1\t': 372, 'SLC25A5\t': 372, 'SDCBP\t': 379, 'UBB\t': 369, 'GMFG\t': 397, 'HMGB1\t': 384, 'COX5B\t': 384, 'PNRC1\t': 332, 'FCGRT\t': 388, 'C1QC\t': 370, 'CDC42\t': 375, 'COX8A\t': 392, 'FKBP1A\t': 373, 'NFKBIA\t': 331, 'BLVRB\t': 395, 'HSP90AA1\t': 385, 'ITGB2\t': 388, 'NPM1\t': 405, 'MAFB\t': 383, 'CEBPD\t': 347, 'PYCARD\t': 401, 'ANXA1\t': 387, 'APOC1\t': 402, 'APOA1\t': 396}


Moramo da transformišemo zatvorene česte skupove (ključeve rečnika) u odgovarajući oblik.

In [78]:
new_dict = {tuple(k.split('\t')[:-1]): v for k, v in patterns_04.items()}

print(new_dict)

{('NOP10',): 339, ('NBEAL1',): 349, ('GLUL',): 333, ('PSMA7',): 328, ('NDUFA4',): 344, ('UQCRQ',): 333, ('SLC25A3',): 340, ('RBM3',): 344, ('HCLS1',): 332, ('POLR2L',): 330, ('COX7B',): 335, ('IFITM3',): 347, ('NAP1L1',): 346, ('USMG5',): 352, ('ASAH1',): 335, ('MT-ND6',): 354, ('CSTB',): 327, ('ARPC5',): 362, ('RNASE6',): 367, ('EDF1',): 342, ('HIGD2A',): 326, ('COX7A2',): 350, ('UBL5',): 344, ('AP2S1',): 342, ('FXYD5',): 347, ('JUN',): 334, ('GLTSCR2',): 345, ('ATP5D',): 358, ('ANXA2',): 345, ('CD68',): 337, ('ZFP36L1',): 337, ('CORO1A',): 356, ('TIMP1',): 388, ('EMP3',): 372, ('CHCHD2',): 355, ('EVI2B',): 355, ('LAMTOR4',): 378, ('PCBP1',): 372, ('SLC25A5',): 372, ('SDCBP',): 379, ('UBB',): 369, ('GMFG',): 397, ('HMGB1',): 384, ('COX5B',): 384, ('PNRC1',): 332, ('FCGRT',): 388, ('C1QC',): 370, ('CDC42',): 375, ('COX8A',): 392, ('FKBP1A',): 373, ('NFKBIA',): 331, ('BLVRB',): 395, ('HSP90AA1',): 385, ('ITGB2',): 388, ('NPM1',): 405, ('MAFB',): 383, ('CEBPD',): 347, ('PYCARD',): 401, (

Ovde definišemo opštu funkciju za analizu po zadatom parametru *minSup*.

In [79]:
def analyze_patterns(minSup):
    obj = charm.CHARM(
    pami_df,
    minSup=minSup,
    sep="\t"
    )

    obj.mine()
    patterns = obj.getPatterns()

    new_patterns = {tuple(k.split('\t')[:-1]): v for k, v in patterns.items()}
    
    pattern_df = pd.DataFrame(
    [(k, v) for k, v in new_patterns.items()],
    columns=["Itemset", "Support Count"]
    )
    pattern_df["Length"] = pattern_df["Itemset"].apply(len)
    
    pattern_df["Length"].value_counts().sort_index()

    print(pattern_df.sort_values("Support Count", ascending=False).head(20))
    print("--------------------------------------------------------------------")
    print(pattern_df.sort_values("Length", ascending=False).head(20))
    

In [80]:
analyze_patterns(0.1)

Closed Frequent patterns were generated successfully using CHARM algorithm
          Itemset  Support Count  Length
7886      (NPM1,)            405       1
7909     (APOC1,)            402       1
7904    (PYCARD,)            401       1
7587      (GMFG,)            397       1
7910     (APOA1,)            396       1
7857     (BLVRB,)            395       1
7813     (COX8A,)            392       1
7746     (FCGRT,)            388       1
6844     (TIMP1,)            388       1
7878     (ITGB2,)            388       1
7907     (ANXA1,)            387       1
7867  (HSP90AA1,)            385       1
7668     (COX5B,)            384       1
7625     (HMGB1,)            384       1
7893      (MAFB,)            383       1
7501     (SDCBP,)            379       1
7271   (LAMTOR4,)            378       1
7784     (CDC42,)            375       1
7829    (FKBP1A,)            373       1
7476   (SLC25A5,)            372       1
----------------------------------------------------------------

In [81]:
analyze_patterns(0.35)

Closed Frequent patterns were generated successfully using CHARM algorithm
        Itemset  Support Count  Length
92      (NPM1,)            405       1
97     (APOC1,)            402       1
95    (PYCARD,)            401       1
78      (GMFG,)            397       1
98     (APOA1,)            396       1
89     (BLVRB,)            395       1
86     (COX8A,)            392       1
66     (TIMP1,)            388       1
83     (FCGRT,)            388       1
91     (ITGB2,)            388       1
96     (ANXA1,)            387       1
90  (HSP90AA1,)            385       1
79     (HMGB1,)            384       1
80     (COX5B,)            384       1
93      (MAFB,)            383       1
76     (SDCBP,)            379       1
72   (LAMTOR4,)            378       1
85     (CDC42,)            375       1
87    (FKBP1A,)            373       1
67      (EMP3,)            372       1
--------------------------------------------------------------------
        Itemset  Support Count  Lengt